In [9]:
import pandas as pd

# 1. Load all three raw datasets using the correct root path (no '../')
cbk_df = pd.read_csv("data/raw/cbk_macro_indicators.csv")
cpi_df = pd.read_csv("data/raw/knbs_cpi_monthly.csv")
maize_df = pd.read_csv("data/raw/maize_market_prices_raw.csv")

# 2. Verify CBK Macroeconomic Indicators
print("=" * 60)
print("1. CBK MACROECONOMIC INDICATORS VERIFICATION")
print("=" * 60)
print(cbk_df.info())
print("\nMissing Values:")
print(cbk_df.isnull().sum())
print("\nSummary Statistics:")
display(cbk_df.describe())

# 3. Verify KNBS CPI Monthly Data
print("\n" + "=" * 60)
print("2. KNBS CPI MONTHLY VERIFICATION")
print("=" * 60)
print(cpi_df.info())
print("\nMissing Values:")
print(cpi_df.isnull().sum())
print("\nSummary Statistics:")
display(cpi_df.describe())

# 4. Verify Maize Market Prices Raw Data
print("\n" + "=" * 60)
print("3. MAIZE MARKET PRICES RAW VERIFICATION")
print("=" * 60)
print(maize_df.info())
print("\nMissing Values:")
print(maize_df.isnull().sum())
print("\nUnique Counties Found:")
print(maize_df["county"].unique() if "county" in maize_df.columns else "No county column found")
print("\nSummary Statistics:")
display(maize_df.describe())

1. CBK MACROECONOMIC INDICATORS VERIFICATION
<class 'pandas.DataFrame'>
RangeIndex: 132 entries, 0 to 131
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   132 non-null    str    
 1   exchange_rate_usd_kes  127 non-null    float64
 2   central_bank_rate      132 non-null    float64
dtypes: float64(2), str(1)
memory usage: 4.5 KB
None

Missing Values:
date                     0
exchange_rate_usd_kes    5
central_bank_rate        0
dtype: int64

Summary Statistics:


,exchange_rate_usd_kes,central_bank_rate
count,127.000000,132.000000
mean,114.311496,10.988636
std,14.807230,1.302473
min,-15.200000,9.500000
25%,107.205000,10.000000
50%,114.340000,10.500000
75%,123.160000,12.500000
max,131.920000,13.000000



2. KNBS CPI MONTHLY VERIFICATION
<class 'pandas.DataFrame'>
RangeIndex: 132 entries, 0 to 131
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   date       132 non-null    str    
 1   cpi_index  128 non-null    float64
dtypes: float64(1), str(1)
memory usage: 3.5 KB
None

Missing Values:
date         0
cpi_index    4
dtype: int64

Summary Statistics:


,cpi_index
count,128.000000
mean,208.076094
std,34.398188
min,150.000000
25%,176.965000
50%,208.810000
75%,239.277500
max,268.000000



3. MAIZE MARKET PRICES RAW VERIFICATION
<class 'pandas.DataFrame'>
RangeIndex: 660 entries, 0 to 659
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   660 non-null    str    
 1   county                 660 non-null    str    
 2   maize_retail_price_kg  645 non-null    float64
dtypes: float64(1), str(2)
memory usage: 28.1 KB
None

Missing Values:
date                      0
county                    0
maize_retail_price_kg    15
dtype: int64

Unique Counties Found:
<ArrowStringArray>
['Nairobi', 'Nakuru', 'Uasin Gishu (Eldoret)', 'Kisumu', 'Mombasa']
Length: 5, dtype: str

Summary Statistics:


,maize_retail_price_kg
count,645.000000
mean,165.610109
std,3060.704289
min,23.310000
25%,38.850000
50%,45.060000
75%,51.830000
max,77777.000000


In [ ]:
import pandas as pd
import numpy as np
import os

os.makedirs("data/processed",exist_ok = True)
print("=" * 60)
print("COMPREHENSIVE DATA CLEANING PIPELINE")
print("=" * 60)


#cleaning the central bank of kenya(CBK) Dataset
print("\n[1/3] Processing CBK Macroeconomic Indicators...")
cbk_df = pd.read_csv("data/raw/cbk_macro_indicators.csv")
cbk_df = cbk_df.drop_duplicates().copy()
cbk_df = cbk_df.dropna(subset=["date"]).copy()
cbk_df["date"] = pd.to_datetime(cbk_df["date"],errors = "coerce")
cbk_df = cbk_df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
cbk_df["exchange_rate_usd_kes"] = cbk_df["exchange_rate_usd_kes"].apply(lambda x: np.nan if x < 0 else x)
cbk_df["exchange_rate_usd_kes"] = (
    cbk_df["exchange_rate_usd_kes"]
    .interpolate(method = 'linear')
    .ffill()
    .bfill()
)
cbk_df["central_bank_rate"] = (
    cbk_df["central_bank_rate"]
    .interpolate(method="linear")
    .ffill()
    .bfill()
)
cbk_path = "data/processed/cbk_macro_indicators_clean.csv"
cbk_df.to_csv(cbk_path, index=False)
print(f" -> Saved to {cbk_path}")
print(f" -> Final Null Count: {cbk_df.isnull().sum().sum()} (Target: 0)")

#cleaning the KNBS consumer price index(CPI) Dataset

print("\n[2/3] Processing KNBS CPI Monthly Data...")
cpi_df = pd.read_csv("data/raw/knbs_cpi_monthly.csv")
cpi_df = cpi_df.drop_duplicates().copy()
cpi_df = cpi_df.dropna(subset=["date"]).copy()
cpi_df["date"] = pd.to_datetime(cpi_df["date"], errors="coerce")
cpi_df = cpi_df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

print("\n[2/3] Processing KNBS CPI Monthly Data...")
cpi_df = pd.read_csv("data/raw/knbs_cpi_monthly.csv")
cpi_df = cpi_df.drop_duplicates().copy()
cpi_df = cpi_df.dropna(subset=["date"]).copy()
cpi_df["date"] = pd.to_datetime(cpi_df["date"], errors="coerce")
cpi_df = cpi_df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

cpi_path = "data/processed/knbs_cpi_monthly_clean.csv"
cpi_df.to_csv(cpi_path, index=False)
print(f" -> Saved to {cpi_path}")
print(f" -> Final Null Count: {cpi_df.isnull().sum().sum()} (Target: 0)")

#cleaning the maize market prices dataset
print("\n[3/3] Processing Maize Market Prices Data...")
maize_df = pd.read_csv("data/raw/maize_market_prices_raw.csv")

maize_df = maize_df.drop_duplicates.copy()
maize_df["date"] = pd.to_datetime(maize_df["date"],errors="coerce")
maize_df["county"] = maize_df["county"].str.strip().str.title()
maize_df["maize_retail_price_kg"] = maize_df["maize_retail_price_kg"].apply(
    lambda x: np.nan if x > 500 or x < 0 else x
)

